<a href="https://colab.research.google.com/github/kreaja/TrialRepositoryJustForMe/blob/main/hw01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 1: Text Classification
## CIS 5300: Natural Language Processing
## Total Points: 107

---

### Overview

In this assignment, you'll build a text classifier to distinguish between **simple** and **complex** words. For example, `heard`, `sat`, and `town` are simple words, while `abdicate`, `detained`, and `vintners` are complex words.

This task — **Complex Word Identification** — is the first step in text simplification, which aims to replace complex words with simpler synonyms. Text simplification is useful for making text accessible to younger readers, English language learners, and people with reading disabilities.

**Labels:** Throughout this homework, **1 = complex word** (positive class) and **0 = simple word** (negative class). When we compute precision and recall, "positive" always means "complex."

### Learning Goals

After completing this homework, you will be able to:

1. **Implement evaluation metrics** (precision, recall, F1) from scratch and understand what they measure *(Module 1, Lectures 2.1–2.3)*
2. **Build baselines** to establish lower bounds on task difficulty *(Module 1, Lecture 1.3)*
3. **Engineer features** from text with principled linguistic reasoning *(Chapter 4, §4.3)*
4. **Train and compare classifiers** using scikit-learn (Naive Bayes, Logistic Regression) *(Module 1, Lectures 1.4–1.5; Chapter 4; Appendix B)*
5. **Perform error analysis** to understand model limitations *(Module 1, Lecture 2.3)*
6. **Evaluate domain generalization** — does your model work on text from different domains? *(Module 1, Lecture 2.3)*

### Deliverables

Submit **all of the following** to Gradescope as a single submission:

| File               | Description                                                        |
|--------------------|--------------------------------------------------------------------|
| `hw01.ipynb`       | This notebook with all code cells completed                        |
| `hw01.py`          | This extracted as a .py file.                                      |
| `hw01_writeup.pdf` | Written answers to all questions marked **[WRITEUP]**              |
| `test_labels.txt`  | Your best model's predictions on the test set (one label per line) |

### Point Breakdown

| Section | Points | Type |
|---------|--------|------|
| 1. Evaluation Metrics | 15 | Code (autograded) |
| 2. Baselines | 26 | Code (autograded) + Writeup |
| 3. Feature Engineering & Classifiers | 28 | Code (autograded) + Writeup |
| 4. Your Own Model & Error Analysis | 18 | Writeup |
| 5. Domain Generalization | 10 | Code + Writeup |
| 6. Reflection Questions | 10 | Writeup |
| **Total** | **107** | |
| Leaderboard bonus (optional) | +5 | Extra credit |

### Recommended Reading

- [Chapter 4: Logistic Regression and Text Classification](https://web.stanford.edu/~jurafsky/slp3/4.pdf) — Jurafsky & Martin, SLP 3rd Edition
- [Appendix B: Naive Bayes](https://web.stanford.edu/~jurafsky/slp3/B.pdf) — Supplementary mathematical details on the Naive Bayes algorithm
- Module 1 Lectures: Text Classification and Sentiment Analysis, Naive Bayes Classifier, Parameter Estimation, Precision/Recall/F-measure, Multiclass Evaluation, Evaluation Methodology

---
## Section 0: Setup

Run the following cells to install dependencies and load the data. **Do not modify these cells.**

In [ ]:
# Install dependencies
!pip install -q scikit-learn numpy matplotlib syllables

In [ ]:
# === Data Download ===
# This cell downloads the homework data from HuggingFace.
# The dataset is a proper HuggingFace dataset with train/validation/test splits.
# See: https://huggingface.co/datasets/CCB/cis5300-text-classification

!pip install -q datasets huggingface_hub

import os
if not os.path.exists('data'):
    os.makedirs('data')

    # Load the structured dataset (train/validation/test splits)
    from datasets import load_dataset
    dataset = load_dataset("CCB/cis5300-text-classification")
    print(f"Loaded dataset: {dataset}")

    # Save as local files for compatibility with the rest of this notebook
    for split_name, file_name in [("train", "complex_words_training.txt"),
                                   ("validation", "complex_words_development.txt")]:
        split = dataset[split_name]
        with open(f"data/{file_name}", "w") as f:
            f.write("WORD\tLABEL\tANNOTATORS\tSENTENCE\tSENTENCE_INDEX\n")
            for row in split:
                f.write(f"{row['word']}\t{row['label']}\t{row['annotators']}\t{row['sentence']}\t{row['sentence_index']}\n")

    # Save test set (no labels)
    test = dataset["test"]
    with open("data/complex_words_test_unlabeled.txt", "w") as f:
        f.write("WORD\tSENTENCE\tSENTENCE_INDEX\n")
        for row in test:
            f.write(f"{row['word']}\t{row['sentence']}\t{row['sentence_index']}\n")

    # Download domain generalization test sets
    bio = load_dataset("CCB/cis5300-text-classification", "biomedical")
    bio["test"].to_csv("data/complex_biomed_test.tsv", sep="\t", index=False)

    news = load_dataset("CCB/cis5300-text-classification", "news")
    news["test"].to_csv("data/cwi2018_news_test.tsv", sep="\t", index=False)

    # Download n-gram counts (supplementary, not a structured dataset)
    from huggingface_hub import hf_hub_download
    hf_hub_download("CCB/cis5300-text-classification", "ngram_counts.txt.gz",
                    repo_type="dataset", local_dir="data")

    print(f"All data downloaded: {len(os.listdir('data'))} files in data/")
else:
    print("Data already downloaded.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import gzip
import syllables

In [ ]:
########## DO NOT MODIFY ##########
# Data loading functions

def load_labeled_file(data_file):
    """Load a labeled dataset (word, label) from a TSV file.
    Returns (words, labels) where words is a list of strings
    and labels is a list of integers (0=simple, 1=complex)."""
    words, labels = [], []
    with open(data_file, 'rt', encoding='utf-8') as f:
        next(f)  # skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                words.append(parts[0].lower())
                labels.append(int(parts[1]))
    return words, labels

def load_unlabeled_file(data_file):
    """Load an unlabeled dataset (words only) from a file.
    Returns a list of strings."""
    with open(data_file, 'rt', encoding='utf-8') as f:
        next(f)  # skip header
        words = [line.strip().split('\t')[0].lower() for line in f if line.strip()]
    return words

def load_ngram_counts(ngram_counts_file='data/ngram_counts.txt.gz'):
    """Load Google NGram word frequency counts.
    Returns a dict mapping lowercase words to their frequency counts (integers)."""
    counts = defaultdict(int)
    with gzip.open(ngram_counts_file, 'rt') as f:
        for line in f:
            token, count = line.strip().split('\t')
            if token[0].islower():
                counts[token] = int(count)
    return counts

In [ ]:
if __name__ == "__main__":
    ########## DO NOT MODIFY ##########
    # Load datasets. Only runs when executed directly (not when imported by autograder).
    try:
        train_words, train_labels = load_labeled_file('data/complex_words_training.txt')
        dev_words, dev_labels = load_labeled_file('data/complex_words_development.txt')
        test_words = load_unlabeled_file('data/complex_words_test_unlabeled.txt')

        # Load word frequency data (this takes ~20 seconds)
        print("Loading Google NGram counts (this takes ~20 seconds)...")
        ngram_counts = load_ngram_counts()
        print(f"Loaded {len(ngram_counts):,} word frequencies.")
    except FileNotFoundError as e:
        print(f"\u26a0\ufe0f  Data files not found: {e}")
        print("   If you're in Colab, run the data download cell above first.")
        train_words, train_labels = [], []
        dev_words, dev_labels = [], []
        test_words = []
        ngram_counts = {}


In [ ]:
if __name__ == "__main__":
    ########## DO NOT MODIFY ##########
    # Explore the data
    if not train_words:
        print("\u26a0\ufe0f  Skipping data exploration: no training data loaded.")
    else:
        n_train = len(train_words)
        n_complex_train = sum(train_labels)
        n_simple_train = n_train - n_complex_train

        print(f"Training set: {n_train} words")
        print(f"  Simple (0): {n_simple_train} ({n_simple_train/n_train:.1%})")
        print(f"  Complex (1): {n_complex_train} ({n_complex_train/n_train:.1%})")
        print(f"\nDev set: {len(dev_words)} words")
        print(f"Test set: {len(test_words)} words (unlabeled)")

        print(f"\nExample simple words: {[w for w, l in zip(train_words[:50], train_labels[:50]) if l == 0][:5]}")
        print(f"Example complex words: {[w for w, l in zip(train_words[:50], train_labels[:50]) if l == 1][:5]}")

        print(f"\n\u26a0\ufe0f Note: The classes are imbalanced \u2014 simple words are the majority.")
        print(f"   This is why accuracy can be misleading (see Lecture M01.2.1).")


---
## Section 1: Evaluation Metrics (15 points)

Before building any classifiers, you need to implement the evaluation metrics we'll use throughout this homework. As discussed in **Lecture M01.2.1**, precision and recall are more informative than accuracy when classes are imbalanced.

### Quick Review: TP, FP, TN, FN

Given two parallel lists — `y_true` (gold labels) and `y_pred` (predicted labels) — where each element is either `0` (simple) or `1` (complex):

| | Predicted Complex (1) | Predicted Simple (0) |
|---|---|---|
| **Actually Complex (1)** | True Positive (TP) | False Negative (FN) |
| **Actually Simple (0)** | False Positive (FP) | True Negative (TN) |

**Worked Example:**
```
y_true = [1, 0, 1, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 1, 1, 0, 1, 0]
                ^     ^
      FN (missed)     FP (false alarm)

TP = 3 (positions 0, 3, 6: correctly predicted complex)
FP = 1 (position 4: predicted complex but actually simple)
FN = 1 (position 2: predicted simple but actually complex)
TN = 3 (positions 1, 5, 7: correctly predicted simple)

Precision = TP / (TP + FP) = 3 / (3 + 1) = 0.75
Recall    = TP / (TP + FN) = 3 / (3 + 1) = 0.75
F1        = 2 * P * R / (P + R) = 2 * 0.75 * 0.75 / (0.75 + 0.75) = 0.75
```

### Problem 1.1 (15 points: code)

Implement `get_precision`, `get_recall`, and `get_fscore`. Each function takes two lists of integers (0s and 1s) of equal length.

**Edge cases:** If the denominator is zero (e.g., no positive predictions), return `0.0`.

In [ ]:
""".   IN CASE NEEDED
import gzip
import os
from collections import defaultdict
from math import log1p

import numpy as np
import matplotlib.pyplot as plt
import syllables

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
"""

def get_precision(y_true, y_pred):
    """Compute precision for binary classification.
    Args:
        y_true: list of int (0 or 1) — gold labels
        y_pred: list of int (0 or 1) — predicted labels
    Returns:
        float: precision = TP / (TP + FP). Returns 0.0 if no positive predictions.
    """
    any_pos = False
    tp, fp = 0, 0
    for truth, pred in zip(y_true, y_pred):
        if truth == 1 and pred == 1:
            tp += 1
            any_pos = True
        elif truth == 0 and pred == 1:
            fp += 1
            any_pos = True
    return 0.0 if not any_pos else tp / (tp + fp)


def get_recall(y_true, y_pred):
    """Compute recall for binary classification.
    Args:
        y_true: list of int (0 or 1) — gold labels
        y_pred: list of int (0 or 1) — predicted labels
    Returns:
        float: recall = TP / (TP + FN). Returns 0.0 if no actual positives.
    """
    any_pos = False
    tp, fn = 0, 0
    for truth, pred in zip(y_true, y_pred):
        if truth == 1 and pred == 1:
            tp += 1
            any_pos = True
        elif truth == 1 and pred == 0:
            fn += 1
            any_pos = True
    return 0.0 if not any_pos else tp / (tp + fn)


def get_fscore(y_true, y_pred):
    """Compute F1 score for binary classification.
    Args:
        y_true: list of int (0 or 1) — gold labels
        y_pred: list of int (0 or 1) — predicted labels
    Returns:
        float: F1 = 2 * P * R / (P + R). Returns 0.0 if P + R = 0.
    """

    p, r = get_precision(y_true, y_pred), get_recall(y_true, y_pred)
    if p + r == 0:
        return 0.0
    else:
        return 2 * p * r / (p + r)

In [ ]:
if __name__ == "__main__":
    # Sanity-check assertions — only run interactively.
    try:
        # Sanity check — verify against the worked example above
        y_true_ex = [1, 0, 1, 1, 0, 0, 1, 0]
        y_pred_ex = [1, 0, 0, 1, 1, 0, 1, 0]

        assert abs(get_precision(y_true_ex, y_pred_ex) - 0.75) < 1e-6, \
            f"Expected precision=0.75, got {get_precision(y_true_ex, y_pred_ex)}"
        assert abs(get_recall(y_true_ex, y_pred_ex) - 0.75) < 1e-6, \
            f"Expected recall=0.75, got {get_recall(y_true_ex, y_pred_ex)}"
        assert abs(get_fscore(y_true_ex, y_pred_ex) - 0.75) < 1e-6, \
            f"Expected F1=0.75, got {get_fscore(y_true_ex, y_pred_ex)}"

        # Edge case: no positive predictions
        assert get_precision([1, 1, 0], [0, 0, 0]) == 0.0, "Precision should be 0 when no positive predictions"
        assert get_recall([0, 0, 0], [1, 1, 1]) == 0.0, "Recall should be 0 when no actual positives"

        print("✓ All sanity checks passed!")
    except Exception as e:
        print(f"\u26a0\ufe0f  Sanity checks skipped: {type(e).__name__}: {e}")


---
## Section 2: Baselines (26 points)

Before using machine learning, we build simple baselines to understand how hard the task is. A model that can't beat simple baselines isn't learning anything useful.

### 2.1 Always-Complex Baseline (8 points: 6 code + 2 writeup)

The simplest possible baseline: predict every word as complex (label = 1).

**Note:** This is NOT the majority class — as you saw above, simple words are the majority (~58%). We predict all-complex because it gives us 100% recall on the positive class, which is a useful reference point.

In [ ]:
def all_complex(words):
    """Baseline that predicts every word as complex.
    Args:
        words: list of str — words to classify
    Returns:
        list of int — predicted labels (all 1s)
    """
    return [1] * len(words)

In [ ]:
if __name__ == "__main__":
    # All-complex baseline evaluation
    try:
        # Evaluate the baseline on training and dev data
        train_pred = all_complex(train_words)
        dev_pred = all_complex(dev_words)

        print("All-Complex Baseline:")
        print(f"  Training — P: {get_precision(train_labels, train_pred):.4f}, "
              f"R: {get_recall(train_labels, train_pred):.4f}, "
              f"F1: {get_fscore(train_labels, train_pred):.4f}")
        print(f"  Dev      — P: {get_precision(dev_labels, dev_pred):.4f}, "
              f"R: {get_recall(dev_labels, dev_pred):.4f}, "
              f"F1: {get_fscore(dev_labels, dev_pred):.4f}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


**[WRITEUP] Answer 2.1 (2 points):** Report the precision, recall, and F1 on both training and dev data. Why is recall 1.0? Why is precision relatively low? What does this tell you about class balance? *(Hint: see Lecture M01.2.1 on the precision-recall tradeoff.)*

### 2.2 Word Length Baseline (9 points: 6 code + 3 writeup)

For this baseline, classify words based on their length. Words with length ≥ threshold are predicted as complex (1), and words with length < threshold are predicted as simple (0).

Your function should:
1. Search over thresholds from 1 to 25 (word lengths rarely exceed 25)
2. For each threshold, compute F1 on the **training data**
3. Select the threshold with the highest training F1
4. Return predictions on the input words using that best threshold

**Expected runtime:** < 5 seconds.

In [ ]:
def word_length_threshold(train_words, train_labels, eval_words):
    """Find the best word-length threshold on training data, then predict on eval_words.

    Decision rule: predict complex (1) if len(word) >= threshold, else simple (0).

    Args:
        train_words: list of str — training words
        train_labels: list of int — training labels (0 or 1)
        eval_words: list of str — words to generate predictions for

    Returns:
        tuple: (eval_predictions, best_threshold)
            eval_predictions: list of int — predicted labels for eval_words
            best_threshold: int — the threshold that maximized training F1
    """
    f1_results = []
    leng_threshs = list(range(1, 26))
    for wlt in leng_threshs:
        preds = [1 if len(word) >= wlt else 0 for word in train_words]
        f1 = get_fscore(train_labels, preds)
        f1_results.append(f1)
    best_threshold = max(leng_threshs, key=lambda x: f1_results[x-1])
    eval_predictions = [1 if len(word) >= best_threshold else 0
                        for word in eval_words]
    # I guessed the F1-optimal cutoff would be 6-8, and was right
    return (eval_predictions, best_threshold)

In [ ]:
if __name__ == "__main__":
    # Word length baseline evaluation
    try:
        # Evaluate word length baseline
        train_pred, best_len_threshold = word_length_threshold(train_words, train_labels, train_words)
        dev_pred, _ = word_length_threshold(train_words, train_labels, dev_words)

        print(f"Best word length threshold: {best_len_threshold}")
        print(f"  Training — P: {get_precision(train_labels, train_pred):.4f}, "
              f"R: {get_recall(train_labels, train_pred):.4f}, "
              f"F1: {get_fscore(train_labels, train_pred):.4f}")
        print(f"  Dev      — P: {get_precision(dev_labels, dev_pred):.4f}, "
              f"R: {get_recall(dev_labels, dev_pred):.4f}, "
              f"F1: {get_fscore(dev_labels, dev_pred):.4f}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


**[WRITEUP] Answer 2.2 (3 points):** Report precision, recall, F1 on training and dev data. Also include:
- The best threshold you found
- A **precision-recall curve** (x-axis = recall, y-axis = precision) showing how P and R change across thresholds 1–25. You can use `matplotlib` to plot this.

In [ ]:
if __name__ == "__main__":
    # Plot precision-recall curve for word length baseline. Note that in these
    # plots, recall is always the x axis and precision the y-axis!

    precisions, recalls = [], []
    leng_threshs = list(range(1, 26))
    # At extreme threshold lengths we get spurious precision values.
    for threshold in leng_threshs:
        preds = [1 if len(word) >= threshold else 0 for word in train_words]
        pr, re = get_precision(train_labels, preds), get_recall(
            train_labels, preds)
        precisions.append(pr)
        recalls.append(re)

    # Now plot (recall, precision) as a point. Connect the points with a line.
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.set_title("Baseline P-R curve, by word length cutoff")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.plot(recalls, precisions, marker='o')
    ax.set(xlim=(0, 1), ylim=(0, 1))
    best_i = leng_threshs.index(best_len_threshold)
    ax.scatter([recalls[best_i]], [precisions[best_i]], s=110,
               facecolors="none", edgecolors="crimson", linewidths=1.8, zorder=3)

    fig.tight_layout()
    plt.show()

### 2.3 Word Frequency Baseline (9 points: 6 code + 3 writeup)

This baseline uses word frequency from the Google NGram corpus. Rare words are more likely to be complex.

**Decision rule:** Predict complex (1) if the word's frequency is **< threshold**, else simple (0). If a word is not in the Google NGram data, treat its frequency as 0 (very rare = likely complex).

**Suggested search range:** Test thresholds from 1 to 100,000,000 (10^8) with a step size of 100,000. This gives 1,000 candidate thresholds and should run in under 2 minutes.

```python
# Suggested range:
for threshold in range(0, 100_000_000, 100_000):
    ...
```

In [ ]:
def word_frequency_threshold(train_words, train_labels, eval_words, ngram_counts):
    """Find the best word-frequency threshold on training data, then predict on eval_words.

    Decision rule: predict complex (1) if ngram_counts[word] < threshold, else simple (0).
    Words not in ngram_counts have frequency 0 (treated as very rare / complex).

    Args:
        train_words: list of str — training words
        train_labels: list of int — training labels (0 or 1)
        eval_words: list of str — words to generate predictions for
        ngram_counts: dict — word -> frequency count

    Returns:
        tuple: (eval_predictions, best_threshold)
            eval_predictions: list of int — predicted labels for eval_words
            best_threshold: int — the threshold that maximized training F1
    """
    # Frequencies as in number-of-occurrences-in-corpus, not proportions/fractions.
    freq_thrs = [100_000 * n for n in range(0, 1001)]
    # Comment: threshold of 0 predicts everything simple (an always-
    # simple baseline), F1 is 0 by construction bc. recall is zero
    f_1_scores = {}
    train_counts = [ngram_counts.get(word, 0) for word in train_words]
    # It's the number wrong and right that matters, so we don't need to keep
    # track of which position corresponds to what word
    for threshold in freq_thrs:
        # Create an array (of length |V_{train}|) of predicted 1's and 0's, ie.
        # complex and simple, respectively
        preds = [1 if ct < threshold else 0 for ct in train_counts]
        f_1_scores[threshold] = get_fscore(train_labels, preds)
    best_freq_threshold = max(freq_thrs, key=lambda x: f_1_scores[x])
    # `get()` is necessary because the eval set may contain words that
    #  are not present in the train set
    eval_preds = [1 if ngram_counts.get(word, 0) < best_freq_threshold
                  else 0 for word in eval_words]
    return (eval_preds, best_freq_threshold)

In [ ]:
if __name__ == "__main__":
    # Word frequency baseline evaluation
    try:
        # Evaluate word frequency baseline
        train_pred, best_freq_threshold = word_frequency_threshold(
            train_words, train_labels, train_words, ngram_counts)
        dev_pred, _ = word_frequency_threshold(
            train_words, train_labels, dev_words, ngram_counts)

        print(f"Best word frequency threshold: {best_freq_threshold:,}")
        print(f"  Training — P: {get_precision(train_labels, train_pred):.4f}, "
              f"R: {get_recall(train_labels, train_pred):.4f}, "
              f"F1: {get_fscore(train_labels, train_pred):.4f}")
        print(f"  Dev      — P: {get_precision(dev_labels, dev_pred):.4f}, "
              f"R: {get_recall(dev_labels, dev_pred):.4f}, "
              f"F1: {get_fscore(dev_labels, dev_pred):.4f}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


In [ ]:
if __name__ == "__main__":
    # Plot precision-recall curve for word frequency baseline
    precisions, recalls = [], []
    freq_thrs = [1] + [100_000 * n for n in range(1, 1001)]
    for threshold in freq_thrs:
        preds = [1 if ngram_counts.get(w, 0) < threshold else 0
                 for w in train_words]
        precisions.append(get_precision(train_labels, preds))
        recalls.append(get_recall(train_labels, preds))
    # Plot (recall, precision) ordered pairs and connect them
    # with a line. Remember the order: (rec, pre)!
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.set_title("Baseline P-R curve by word **frequency** cutoff")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    # Tiny marker for visibility's sake
    ax.plot(recalls, precisions, marker='.', markersize=3)
    # Determine programmatically which point corresponds to optimum threshold
    f1scores = [2 * p * r / (p + r) for p, r in zip(precisions, recalls)]
    # I checked, and p + r is never equal to 0
    best_thrs = max(range(len(freq_thrs)), key=lambda i: f1scores[i])
    ax.scatter([recalls[best_thrs]], [precisions[best_thrs]], s=110, facecolors=
                "none", edgecolors="crimson", linewidths=1.8, zorder=3)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    fig.tight_layout()
    plt.show()

**[WRITEUP] Answer 2.3 (3 points):** Report precision, recall, F1 on training and dev data, the best threshold, and a precision-recall curve. Compare the word length and word frequency baselines: which performs better and why? *(Hint: think about what each feature captures about word complexity.)*

---
## Section 3: Feature Engineering & Classifiers (28 points)

Now let's use scikit-learn classifiers with proper feature engineering. We'll extract three features for each word:

1. **Word length** (number of characters)
2. **Word frequency** (Google NGram count; 0 if not found)
3. **Syllable count** (using the `syllables` library)

### 3.0 Build Feature Vectors (22 points: code)

Implement feature extraction with **z-score normalization**. This is important because word length (1–25) and word frequency (0–billions) are on vastly different scales.

**⚠️ CRITICAL: Always compute mean and standard deviation from the TRAINING data only.** Using dev/test statistics is **data leakage** and will give you misleadingly good results that won't generalize. See Chapter 4, §4.3 on feature scaling.

In [ ]:
def get_features(words, ngram_counts):
    """Extract raw (unnormalized) features for a list of words.
    Features per word:
        0: word length (int)
        1: word frequency from ngram_counts (int, 0 if not found)
        2: syllable count (int)
    Args:
        words: list of str
        ngram_counts: dict — word -> frequency count

    Returns:
        np.ndarray of shape (len(words), 3)
    """
    features = []
    for word in words:
        # Extract the 3 features for this word
        features.append([len(word), ngram_counts.get(word, 0),
                         syllables.estimate(word)])
    return np.array(features, dtype=float)


def normalize_features(X_train, X_eval=None):
    """Apply z-score normalization using training statistics.

    Computes mean and std from X_train, then normalizes both X_train and X_eval.
    If std is 0 for any feature, uses 1.0 to avoid division by zero.
    Args:
        X_train: np.ndarray of shape (m, n) — training features
        X_eval: np.ndarray of shape (m2, n) — eval features (optional)
    Returns:
        If X_eval is None: (X_train_scaled, means, stds)
        If X_eval is provided: (X_train_scaled, X_eval_scaled, means, stds)
    """

    # 1. Compute mean and std from X_train (per feature)
    means = X_train.mean(axis=0)
    stdevs = X_train.std(axis=0)

    # 2. Replace any std == 0 with 1.0 to avoid NaN
    stdevs = np.where(stdevs == 0, 1.0, stdevs)
    # In real life if a feature's values have standard deviation 0 it means
    # they're all identical, making the feature (a column of zeroes once
    # normalized) meaningless -- it should be dropped.

    # 3. Normalize: X_scaled = (X - mean) / std
    X_train_scaled = X_train - means            # element-wise subtraction,
    X_train_scaled = X_train_scaled / stdevs    # two arrays of the same length
    if X_eval is None:
        return (X_train_scaled, means, stdevs)

    # 4. Apply SAME mean and std to X_eval if provided
    X_eval_scaled = X_eval - means  # We shift and shrink X_eval's points using
    X_eval_scaled = X_eval_scaled / stdevs  # *X_train's* mean & std, avoiding
    return (X_train_scaled, X_eval_scaled, means, stdevs)  # data leakage ヅ

In [ ]:
if __name__ == "__main__":
    # Feature matrix construction
    try:
        # Build feature matrices
        X_train_raw = get_features(train_words, ngram_counts)
        X_dev_raw = get_features(dev_words, ngram_counts)

        X_train, X_dev, means, stds = normalize_features(X_train_raw, X_dev_raw)
        Y_train = np.array(train_labels)
        Y_dev = np.array(dev_labels)

        print(f"Training features shape: {X_train.shape}")
        print(f"Dev features shape: {X_dev.shape}")
        print(f"Feature means (from training): {means}")
        print(f"Feature stds (from training): {stds}")
        print(f"\nTraining features mean after normalization: {X_train.mean(axis=0)}")
        print(f"Training features std after normalization: {X_train.std(axis=0)}")

        # Verify normalization
        assert X_train.shape == (len(train_words), 3), f"Expected shape ({len(train_words)}, 3), got {X_train.shape}"
        assert np.allclose(X_train.mean(axis=0), 0, atol=1e-10), "Training features should have mean ≈ 0 after normalization"
        print("\n✓ Feature shapes and normalization look correct!")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


### 3.1 Naive Bayes Classifier

Train a Gaussian Naive Bayes classifier using the normalized features. See **Lecture M01.1.4** for the theory and [sklearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) for the API.

**Why Gaussian NB?** Our features are continuous (not word counts), so we use `GaussianNB` which models each feature as a Gaussian distribution per class. The "naive" assumption is that features are independent given the class — see **Lecture M01.1.4** for why this assumption is often violated but still works well in practice.

In [ ]:
def naive_bayes(X_train, Y_train, X_eval):
    """Train a Gaussian Naive Bayes classifier and predict.

    Args:
        X_train: np.ndarray of shape (m, n) — normalized training features
        Y_train: np.ndarray of shape (m,) — training labels
        X_eval: np.ndarray of shape (m2, n) — normalized eval features

    Returns:
        tuple: (train_predictions, eval_predictions) — both np.ndarray of int
    """
    ## YOUR CODE HERE
    nb_clf = GaussianNB()
    nb_clf.fit(X_train, Y_train)
    train_eval_prdctns = (nb_clf.predict(X_train), nb_clf.predict(X_eval))
    return train_eval_prdctns

In [ ]:
if __name__ == "__main__":
    # Naive Bayes evaluation
    try:
        train_pred, dev_pred = naive_bayes(X_train, Y_train, X_dev)

        print("Naive Bayes:")
        print(f"  Training — P: {get_precision(train_labels, train_pred):.4f}, "
              f"R: {get_recall(train_labels, train_pred):.4f}, "
              f"F1: {get_fscore(train_labels, train_pred):.4f}")
        print(f"  Dev      — P: {get_precision(dev_labels, dev_pred):.4f}, "
              f"R: {get_recall(dev_labels, dev_pred):.4f}, "
              f"F1: {get_fscore(dev_labels, dev_pred):.4f}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


### 3.2 Logistic Regression

Train a Logistic Regression classifier using the same normalized features. See **Chapter 4** for the math and [sklearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for the API.

In [ ]:
def logistic_regression(X_train, Y_train, X_eval):
    """Train a Logistic Regression classifier and predict.

    Args:
        X_train: np.ndarray of shape (m, n) — normalized training features
        Y_train: np.ndarray of shape (m,) — training labels
        X_eval: np.ndarray of shape (m2, n) — normalized eval features

    Returns:
        tuple: (train_predictions, eval_predictions) — both np.ndarray of int
    """
    global lr_coeffs, lr_intercepts
    # This model will be used again in question 6.2, so keep the above in scope

    clf = LogisticRegression(solver='lbfgs', max_iter=1000)
    clf.fit(X_train, Y_train)
    train_eval_prdctns = clf.predict(X_train), clf.predict(X_eval)
    lr_coeffs = clf.coef_
    lr_intercepts = clf.intercept_
    return train_eval_prdctns

In [ ]:
if __name__ == "__main__":
    # Logistic Regression evaluation
    try:
        train_pred, dev_pred = logistic_regression(X_train, Y_train, X_dev)

        print("Logistic Regression:")
        print(f"  Training — P: {get_precision(train_labels, train_pred):.4f}, "
              f"R: {get_recall(train_labels, train_pred):.4f}, "
              f"F1: {get_fscore(train_labels, train_pred):.4f}")
        print(f"  Dev      — P: {get_precision(dev_labels, dev_pred):.4f}, "
              f"R: {get_recall(dev_labels, dev_pred):.4f}, "
              f"F1: {get_fscore(dev_labels, dev_pred):.4f}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Skipped: {type(e).__name__}: {e}")


### 3.3 Classifier Comparison (6 points) [WRITEUP]

**[WRITEUP] Answer 3.3:** Compare Naive Bayes and Logistic Regression. Your answer should address:

1. Report precision, recall, and F1 for both classifiers on both training and dev data.
2. Which model performs better on this task? Is the difference large or small?
3. **Why** might one model outperform the other? Consider:
   - Naive Bayes assumes features are **conditionally independent** given the class. Is this realistic for our features (length, frequency, syllables)? *(Hint: Are word length and syllable count correlated?)*
   - Logistic Regression can learn **feature interactions** through its weight vector. How might this help?
4. How do both ML classifiers compare to the baselines from Section 2?

---
## Section 4: Build Your Own Model (18 points)

Design and train your own classifier for complex word identification. You have creative freedom here, but must follow these requirements:

### Requirements

1. **Use at least one classifier type** not already used (i.e., not `GaussianNB` or `LogisticRegression`). Options include: `SVM`, `RandomForestClassifier`, `GradientBoostingClassifier`, `DecisionTreeClassifier`, etc.
2. **Add at least 3 new features** beyond length, frequency, and syllable count. For each feature, you must provide a **linguistic justification** (see 4.1 below).
3. **Achieve a higher dev F1** than all baselines (Section 2) and both sklearn classifiers (Section 3).

### Important

- **Build your feature extraction in this section** — do NOT modify `get_features()` in Section 3, as that would change your Section 3 results.
- You may use any Python libraries available via pip.
- Train only on the training data. While evaluation for model selection should primarily use the dev data, track performance on both training and dev data for your analysis and report.
- **Troubleshooting**: If your autograder fails to import your .py file, ensure that all additional code that you write particularly in the remaining sections, is placed under the provided `if __name__ == "__main__"` blocks.

### 4.1 Feature Design & Justification (10 points) [WRITEUP]

For each new feature you add (at least 3), fill in the following template in your writeup:

| | Feature 1 | Feature 2 | Feature 3 | ... |
|---|---|---|---|---|
| **Feature name** | e.g., "log frequency" | e.g., "number of WordNet synsets" | e.g., "vowel ratio" | |
| **What linguistic property it captures** | e.g., "Captures diminishing returns of frequency — the difference between freq 100 and 1000 matters more than between 1M and 1.1M" | e.g., "Words with many meanings (polysemous) tend to be common, everyday words" | e.g., "Complex words often have Latin/Greek roots with more consonant clusters" | |
| **Predicted effect on precision vs recall** | e.g., "Should improve both — log scale better separates rare from common words" | e.g., "Should increase precision — fewer false positives on polysemous simple words" | e.g., "May increase recall — catches some complex words that length alone misses" | |
| **After training: did it help?** | e.g., "Yes, F1 improved from 0.72 to 0.74" | e.g., "Marginal improvement (+0.005)" | e.g., "No measurable improvement" | |
| **Why or why not?** | e.g., "Log transformation reduced the dominance of frequency outliers" | e.g., "Most words in our dataset have few synsets regardless of complexity" | e.g., "Vowel ratio doesn't distinguish Latin-root complex words from simple ones" | |

**Some feature ideas** (you are not limited to these):
- Number of [WordNet](https://www.nltk.org/howto/wordnet.html) synonyms or senses
- Whether the word contains common prefixes/suffixes (un-, re-, -tion, -ment, -ous)
- Character-level features (proportion of vowels, presence of rare letter combinations like "ph", "ght")
- Log-transformed frequency
- Whether the word appears in a basic English word list (e.g., [Ogden's Basic English](https://en.wikipedia.org/wiki/Basic_English))
- Number of morphemes (prefix + root + suffix)
- Character bigram probability (how "English-looking" is the word?)

In [ ]:
## I'm going to use Gradient Boosting classifier because it's new to me
# I liked number of syllables. Number of WordNet synonyms sounds promising.
# Definitely replace the frequency with its log-transform. Maybe consider
# membership in Ogden's Basic English, a list which is available online. And
# finally, sure, a rough list of common affixes in the language.

# Ogden's: all-lower-case, note both US/British spellings of 'plow' and 'gray'

OGDEN_852_LOWER = frozenset({
    "come", "get", "give", "go", "keep", "let", "make", "put", "seem",
    "take", "be", "do", "have", "say", "see", "send", "may", "will",
    "about", "across", "after", "against", "among", "at", "before",
    "between", "by", "down", "from", "in", "off", "on", "over", "through",
    "to", "under", "up", "with", "as", "for", "of", "till", "than", "a",
    "the", "all", "any", "every", "little", "much", "no", "other", "some",
    "such", "that", "this", "i", "he", "you", "who", "and", "because",
    "but", "or", "if", "though", "while", "how", "when", "where", "why",
    "again", "ever", "far", "forward", "here", "near", "now", "out",
    "still", "then", "there", "together", "well", "almost", "enough",
    "even", "not", "only", "quite", "so", "very", "tomorrow", "yesterday",
    "north", "south", "east", "west", "please", "yes", "account", "act",
    "addition", "adjustment", "advertisement", "agreement", "air", "amount",
    "amusement", "animal", "answer", "apparatus", "approval", "argument",
    "art", "attack", "attempt", "attention", "attraction", "authority",
    "back", "balance", "base", "behavior", "belief", "birth", "bit", "bite",
    "blood", "blow", "body", "brass", "bread", "breath", "brother",
    "building", "burn", "burst", "business", "butter", "canvas", "care",
    "cause", "chalk", "chance", "change", "cloth", "coal", "color",
    "comfort", "committee", "company", "comparison", "competition",
    "condition", "connection", "control", "cook", "copper", "copy", "cork",
    "cotton", "cough", "country", "cover", "crack", "credit", "crime",
    "crush", "cry", "current", "curve", "damage", "danger", "daughter",
    "day", "death", "debt", "decision", "degree", "design", "desire",
    "destruction", "detail", "development", "digestion", "direction",
    "discovery", "discussion", "disease", "disgust", "distance",
    "distribution", "division", "doubt", "drink", "driving", "dust",
    "earth", "edge", "education", "effect", "end", "error", "event",
    "example", "exchange", "existence", "expansion", "experience", "expert",
    "fact", "fall", "family", "father", "fear", "feeling", "fiction",
    "field", "fight", "fire", "flame", "flight", "flower", "fold", "food",
    "force", "form", "friend", "front", "fruit", "glass", "gold",
    "government", "grain", "grass", "grip", "group", "growth", "guide",
    "harbor", "harmony", "hate", "hearing", "heat", "help", "history",
    "hole", "hope", "hour", "humor", "ice", "idea", "impulse", "increase",
    "industry", "ink", "insect", "instrument", "insurance", "interest",
    "invention", "iron", "jelly", "join", "journey", "judge", "jump",
    "kick", "kiss", "knowledge", "land", "language", "laugh", "law", "lead",
    "learning", "leather", "letter", "level", "lift", "light", "limit",
    "linen", "liquid", "list", "look", "loss", "love", "machine", "man",
    "manager", "mark", "market", "mass", "meal", "measure", "meat",
    "meeting", "memory", "metal", "middle", "milk", "mind", "mine",
    "minute", "mist", "money", "month", "morning", "mother", "motion",
    "mountain", "move", "music", "name", "nation", "need", "news", "night",
    "noise", "note", "number", "observation", "offer", "oil", "operation",
    "opinion", "order", "organization", "ornament", "owner", "page", "pain",
    "paint", "paper", "part", "paste", "payment", "peace", "person",
    "place", "plant", "play", "pleasure", "point", "poison", "polish",
    "porter", "position", "powder", "power", "price", "print", "process",
    "produce", "profit", "property", "prose", "protest", "pull",
    "punishment", "purpose", "push", "quality", "question", "rain", "range",
    "rate", "ray", "reaction", "reading", "reason", "record", "regret",
    "relation", "religion", "representative", "request", "respect", "rest",
    "reward", "rhythm", "rice", "river", "road", "roll", "room", "rub",
    "rule", "run", "salt", "sand", "scale", "science", "sea", "seat",
    "secretary", "selection", "self", "sense", "servant", "sex", "shade",
    "shake", "shame", "shock", "side", "sign", "silk", "silver", "sister",
    "size", "sky", "sleep", "slip", "slope", "smash", "smell", "smile",
    "smoke", "sneeze", "snow", "soap", "society", "son", "song", "sort",
    "sound", "soup", "space", "stage", "start", "statement", "steam",
    "steel", "step", "stitch", "stone", "stop", "story", "stretch",
    "structure", "substance", "sugar", "suggestion", "summer", "support",
    "surprise", "swim", "system", "talk", "taste", "tax", "teaching",
    "tendency", "test", "theory", "thing", "thought", "thunder", "time",
    "tin", "top", "touch", "trade", "transport", "trick", "trouble", "turn",
    "twist", "unit", "use", "value", "verse", "vessel", "view", "voice",
    "walk", "war", "wash", "waste", "water", "wave", "wax", "way",
    "weather", "week", "weight", "wind", "wine", "winter", "woman", "wood",
    "wool", "word", "work", "wound", "writing", "year", "angle", "ant",
    "apple", "arch", "arm", "army", "baby", "bag", "ball", "band", "basin",
    "basket", "bath", "bed", "bee", "bell", "berry", "bird", "blade",
    "board", "boat", "bone", "book", "boot", "bottle", "box", "boy",
    "brain", "brake", "branch", "brick", "bridge", "brush", "bucket",
    "bulb", "button", "cake", "camera", "card", "cart", "carriage", "cat",
    "chain", "cheese", "chest", "chin", "church", "circle", "clock",
    "cloud", "coat", "collar", "comb", "cord", "cow", "cup", "curtain",
    "cushion", "dog", "door", "drain", "drawer", "dress", "drop", "ear",
    "egg", "engine", "eye", "face", "farm", "feather", "finger", "fish",
    "flag", "floor", "fly", "foot", "fork", "fowl", "frame", "garden",
    "girl", "glove", "goat", "gun", "hair", "hammer", "hand", "hat", "head",
    "heart", "hook", "horn", "horse", "hospital", "house", "island",
    "jewel", "kettle", "key", "knee", "knife", "knot", "leaf", "leg",
    "library", "line", "lip", "lock", "map", "match", "monkey", "moon",
    "mouth", "muscle", "nail", "neck", "needle", "nerve", "net", "nose",
    "nut", "office", "orange", "oven", "parcel", "pen", "pencil", "picture",
    "pig", "pin", "pipe", "plane", "plate", "plough", "plow", "pocket",
    "pot", "potato", "prison", "pump", "rail", "rat", "receipt", "ring",
    "rod", "roof", "root", "sail", "school", "scissors", "screw", "seed",
    "sheep", "shelf", "ship", "shirt", "shoe", "skin", "skirt", "snake",
    "sock", "spade", "sponge", "spoon", "spring", "square", "stamp", "star",
    "station", "stem", "stick", "stocking", "stomach", "store", "street",
    "sun", "table", "tail", "thread", "throat", "thumb", "ticket", "toe",
    "tongue", "tooth", "town", "train", "tray", "tree", "trousers",
    "umbrella", "wall", "watch", "wheel", "whip", "whistle", "window",
    "wing", "wire", "worm", "able", "acid", "angry", "automatic",
    "beautiful", "black", "boiling", "bright", "broken", "brown", "cheap",
    "chemical", "chief", "clean", "clear", "common", "complex", "conscious",
    "cut", "deep", "dependent", "early", "elastic", "electric", "equal",
    "fat", "fertile", "first", "fixed", "flat", "free", "frequent", "full",
    "general", "good", "great", "grey", "gray", "hanging", "happy", "hard",
    "healthy", "high", "hollow", "important", "kind", "like", "living",
    "long", "male", "married", "material", "medical", "military", "natural",
    "necessary", "new", "normal", "open", "parallel", "past", "physical",
    "political", "poor", "possible", "present", "private", "probable",
    "quick", "quiet", "ready", "red", "regular", "responsible", "right",
    "round", "same", "second", "separate", "serious", "sharp", "smooth",
    "sticky", "stiff", "straight", "strong", "sudden", "sweet", "tall",
    "thick", "tight", "tired", "true", "violent", "waiting", "warm", "wet",
    "wide", "wise", "yellow", "young", "awake", "bad", "bent", "bitter",
    "blue", "certain", "cold", "complete", "cruel", "dark", "dead", "dear",
    "delicate", "different", "dirty", "dry", "false", "feeble", "female",
    "foolish", "future", "green", "ill", "last", "late", "left", "loose",
    "loud", "low", "mixed", "narrow", "old", "opposite", "public", "rough",
    "sad", "safe", "secret", "short", "shut", "simple", "slow", "small",
    "soft", "solid", "special", "strange", "thin", "white", "wrong",
})

# Common affixes, from Wiktionary: there are going to be false-
# positive matches but I don't expect the magnitude of this last
# feature to be great anyway. Trying to keep it to ones that are
# genuinely **derivational** rather than **inflectional**; so
# no -s, -ed, or -ing.
prefixes = (
    "un", "re", "dis", "in", "im", "il", "ir", "non", "pre", "post",
    "anti", "de", "sub", "super", "trans", "inter", "intra", "over",
    "under", "mis", "semi", "multi", "hyper", "hypo", "micro", "macro",
    "mono", "poly", "auto", "co", "counter", "contra", "circum", "extra",
    "retro", "ultra",
)
suffixes = (
    "tion", "sion", "ation", "ition", "ment", "ness", "ous", "ity", "ety",
    "ance", "ence", "ancy", "ency", "ism", "ist", "ize", "ise", "ify", "ative",
    "ive", "able", "ible", "ary", "ory", "ology", "graphy", "itis", "osis",
    "esque", "escent", "ously", "ically",
)

# Per https://edstem.org/us/courses/102377/discussion/8240832, "Regarding the
# baseline table when doing the delta-F1, it's actually with respect to using
# your chosen new model with only the base 3 features. The intent was to build
# upon those original 3 features, so you'd have 4 features after +1, 5 features
# after +2, and so on. However, if you decide to drop/replace, there's not an
# issue with that as long as that's communicated clearly."

if __name__ == "__main__":
    import nltk
    from nltk.corpus import wordnet as wn
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet')

    # 1. Define your custom feature extraction function
    def custom_feat_extraction(words):
        # The thing I don't like is that the F1-deltas for each custom feature
        # depends on what order we add them in; I would have preferred to compare
        # the baseline-3 to the baseline-3-plus-each custom-feature, 1 at a time.
        features = []
        lemmatizer = WordNetLemmatizer()
        for word in words:  # feats_4_word will be a 6-item list
            feats_4_word = [len(word)]
            feats_4_word.append(log1p(ngram_counts.get(word, 0)))
            # log1p is zero-safe, unlike log which will raise if it's fed 0
            feats_4_word.append(syllables.estimate(word))
            feats_4_word.append(len(wn.synsets(word)))
            # Ogden's list is base forms only -- "kept" won't match "keep",
            # "went" won't match "go". We have to run each word from the
            # dataset through WordNetLemmatizer first. In fact to be thorough
            # I'll try both surface form and lemma, as both verb and noun.
            forms = [word]
            forms.append(lemmatizer.lemmatize(word, pos='v'))
            forms.append(lemmatizer.lemmatize(word, pos='n'))
            feats_4_word.append(
                1 if any(form in OGDEN_852_LOWER for form in forms) else 0)
            # Prefixes/suffixes: 1 point if one present, 2 points if both
            g = 0
            if any(word.startswith(prefix) for prefix in prefixes):
                g += 1
            if any(word.endswith(suffix) for suffix in suffixes):
                g += 1
            feats_4_word.append(g)
            # Finally, tack this list to the end of `features`
            features.append(feats_4_word)
        all_feats = np.array(features, dtype=float)
        return all_feats

    # 2. Build feature matrices for training and dev data
    training_features = custom_feat_extraction(train_words)
    dev_features = custom_feat_extraction(dev_words)

    # 3. Normalize features (using training word set only!)
    custom_train_all, custom_dev_all, mus, sigmas = normalize_features(
        training_features, dev_features)
    # We scale BOTH custom_train and custom_dev with the train data, no info leak

    # 4. Train your classifier
    gbc_all = GradientBoostingClassifier(random_state=300)
    gbc_all.fit(custom_train_all, train_labels)

    # Train a GradientBoosting model using the three features from the previous
    # section, which is the baseline we are supposed to use for the delta comparisons
    gbc_baseline = GradientBoostingClassifier(random_state=300)
    gbc_baseline.fit(X_train, train_labels)              # X_train and X_dev are
    bl_train_pred = gbc_baseline.predict(X_train)        # still in scope from
    bl_dev_pred = gbc_baseline.predict(X_dev)            # before.

    ## Train sub-models with subsets of features. First, length+log_freq+syllables:
    gbc_length_logfreq_syllab = GradientBoostingClassifier(random_state=300).fit(
        custom_train_all[:, 0:3], train_labels)
    # Next add # of WN synonyms
    gbc_synsets = GradientBoostingClassifier(random_state=300).fit(
        custom_train_all[:, 0:4], train_labels)
    # Add Ogden membership (0/1)
    gbc_ogden = GradientBoostingClassifier(random_state=300).fit(
        custom_train_all[:, 0:5], train_labels)

    # Used in the table in the writeup
    custom_train_pred = gbc_all.predict(custom_train_all)
    custom_train_llfs_pred = gbc_length_logfreq_syllab.predict(
        custom_train_all[:, :3])
    custom_train_synsets_pred = gbc_synsets.predict(custom_train_all[:, :4])
    custom_train_ogden_pred = gbc_ogden.predict(custom_train_all[:, :5])

    # 5. Evaluate on dev data
    custom_dev_pred = gbc_all.predict(custom_dev_all)
    custom_dev_llfs_pred = gbc_length_logfreq_syllab.predict(custom_dev_all[:, :3])
    custom_dev_synsets_pred = gbc_synsets.predict(custom_dev_all[:, :4])
    custom_dev_ogden_pred = gbc_ogden.predict(custom_dev_all[:, :5])

    # Report results:
    print(f"GradientBoostingClassifier with 3-feature baseline:")
    print(f"Train — P: {get_precision(train_labels, bl_train_pred):.4f}, "
           f"R: {get_recall(train_labels, bl_train_pred):.4f}, "
           f"F1: {get_fscore(train_labels, bl_train_pred):.4f}")
    print(f"GBC with len/log-freq/syllables only:")
    print(f"Train — P: {get_precision(train_labels, custom_train_llfs_pred):.4f}, "
           f"R: {get_recall(train_labels, custom_train_llfs_pred):.4f}, "
           f"F1: {get_fscore(train_labels, custom_train_llfs_pred):.4f}")
    print(f"GBC with len/log-freq/syllables + # synonyms:")
    print(f"Train — P: {get_precision(train_labels, custom_train_synsets_pred):.4f}, "
           f"R: {get_recall(train_labels, custom_train_synsets_pred):.4f}, "
           f"F1: {get_fscore(train_labels, custom_train_synsets_pred):.4f}")
    print(f"GBC with first five features:")
    print(f"Train — P: {get_precision(train_labels, custom_train_ogden_pred):.4f}, "
           f"R: {get_recall(train_labels, custom_train_ogden_pred):.4f}, "
           f"F1: {get_fscore(train_labels, custom_train_ogden_pred):.4f}")
    print(f"Custom model with all six features:")
    print(f"Train — P: {get_precision(train_labels, custom_train_pred):.4f}, "
           f"R: {get_recall(train_labels, custom_train_pred):.4f}, "
           f"F1: {get_fscore(train_labels, custom_train_pred):.4f}")

    print(f"GradientBoostingClassifier with 3-feature baseline:")
    print(f"  Dev — P: {get_precision(dev_labels, bl_dev_pred):.4f}, "
           f"R: {get_recall(dev_labels, bl_dev_pred):.4f}, "
           f"F1: {get_fscore(dev_labels, bl_dev_pred):.4f}")
    print(f"GBC with len/log-freq/syllables only:")
    print(f"  Dev — P: {get_precision(dev_labels, custom_dev_llfs_pred):.4f}, "
           f"R: {get_recall(dev_labels, custom_dev_llfs_pred):.4f}, "
           f"F1: {get_fscore(dev_labels, custom_dev_llfs_pred):.4f}")
    print(f"GBC with len/log-freq/syllables + # synonyms:")
    print(f"  Dev — P: {get_precision(dev_labels, custom_dev_synsets_pred):.4f}, "
           f"R: {get_recall(dev_labels, custom_dev_synsets_pred):.4f}, "
           f"F1: {get_fscore(dev_labels, custom_dev_synsets_pred):.4f}")
    print(f"GBC with first five features:")
    print(f"  Dev — P: {get_precision(dev_labels, custom_dev_ogden_pred):.4f}, "
           f"R: {get_recall(dev_labels, custom_dev_ogden_pred):.4f}, "
           f"F1: {get_fscore(dev_labels, custom_dev_ogden_pred):.4f}")
    dev_p = get_precision(dev_labels, custom_dev_pred)
    dev_r = get_recall(dev_labels, custom_dev_pred)
    dev_f1 = get_fscore(dev_labels, custom_dev_pred)  # Used in part 5
    print(f"Custom model with all six features:")
    print(f"  Dev — P: {dev_p:.4f}, R: {dev_r:.4f}, F1: {dev_f1:.4f}")

**[WRITEUP] Answer 4.1:** Fill in the feature justification table above for your 3+ features. Also report:
- What classifier(s) did you try? Which performed best?
- Report precision, recall, and F1 on training and dev data incrementally as you add each feature.
- How does your model compare to the baselines and sklearn classifiers?

### 4.2 Error Analysis (8 points) [WRITEUP]

**[WRITEUP] Answer 4.2:** Perform a detailed error analysis of your best model:

1. Give **3 examples** of words your model correctly classifies and explain why it gets them right.
2. Give **3 examples** of words your model incorrectly classifies.
3. Identify **at least two categories of errors** your model makes (e.g., "fails on short but obscure words", "marks common long words as complex"). For each category:
   - Describe the pattern
   - Give 2+ specific examples
   - Explain **why** your model/features fail on this category
   - Suggest a feature or approach that might fix it

In [ ]:
if __name__ == "__main__":
    # True positives and true negatives
    tps = [w for w, true, pred
           in zip(dev_words, dev_labels, custom_dev_pred)
           if true == 1 and pred == 1]
    tns = [w for w, true, pred
           in zip(dev_words, dev_labels, custom_dev_pred)
           if true == 0 and pred == 0]
    # Find false positives and false negatives
    false_positives = [w for w, true, pred
                       in zip(dev_words, dev_labels, custom_dev_pred)
                       if true == 0 and pred == 1]
    false_negatives = [w for w, true, pred
                       in zip(dev_words, dev_labels, custom_dev_pred)
                       if true == 1 and pred == 0]
    print(f"True positives (predicted complex, actually complex): "
          f"{tps[:10]}")
    print(f"True negatives (predicted simple, actually simple): "
          f"{tns[:10]}")
    print(f"False positives (predicted complex, actually simple): "
          f"{false_positives[:]}")
    print(f"False negatives (predicted simple, actually complex): "
          f"{false_negatives[:]}")
    # Why is "scam" considered gold-complex?

### 4.3 Test Set Predictions

Generate predictions for the unlabeled test set and save them to `test_labels.txt`. This file should contain one label per line (0 or 1), in the same order as the test words.

**Reminder:** The test set is stored in `test_words`. Use your `get_features` function and normalize with the **same** training mean/std before predicting.

**Do NOT shuffle the test words.** The order must match the original file.

In [ ]:
if __name__ == "__main__":
    ## The "data/complex_words_test_unlabeled.txt" was loaded into variable
    ## `test_words` at the beginning.
    test_feats_all = custom_feat_extraction(test_words)
    _, standardized_test_feats, _, _ = normalize_features(training_features, test_feats_all)
    test_pred = gbc_all.predict(standardized_test_feats)
    # Save predictions
    with open('test_labels.txt', 'w') as f:
        f.write('\n'.join(str(label) for label in test_pred))
    print(f"Saved {len(test_pred)} predictions to test_labels.txt")

### 4.4 Leaderboard (Optional, +5 bonus points)

Submit your `test_labels.txt` to the class leaderboard on Gradescope. The top 3 submissions receive 5 bonus points.

For the leaderboard, you may use **any additional data or resources** (e.g., pre-trained embeddings, external word lists, transformer models). Just document what you used in your writeup.

---
## Section 5: Domain Generalization (10 points)

Your model was trained on general English text. But would it work on text from a completely different domain? In this section, you'll evaluate your best model on two external Complex Word Identification datasets to test how well your features generalize.

This connects to a key lesson from **the Evaluation Methodology lecture**: in-distribution performance can be misleading. A model that works well on your dev set may fail on text from a different source.

### 5.1 Load External Datasets

We provide two labeled CWI datasets from different domains:

1. **Biomedical** (`data/complex_biomed_test.tsv`) — Words from PubMed biomedical abstracts, labeled for complexity by crowdworkers. From the CompLex dataset (Shardlow et al., 2020). 289 words.

2. **News** (`data/cwi2018_news_test.tsv`) — Words from news articles, labeled for complexity by both native and non-native English speakers. From the SemEval 2018 CWI Shared Task. 1,813 words.

Load both datasets and compute the class distribution (% complex) for each. **Report these values in Answer 5.3.**

In [ ]:
if __name__ == "__main__":
    # Each file is a TSV with columns: word, complexity_score (or complexity_prob), label

    def load_external_dataset(filepath):
        words, labels = [], []
        with open(filepath, 'rt', encoding='utf-8') as f:
            next(f)  # skip header
            for line in f:
                parts = line.strip().split('\t')
                words.append(parts[0].lower())
                labels.append(int(parts[2]))  # binary label column
        return words, labels

    biomed_words, biomed_labels = load_external_dataset('data/complex_biomed_test.tsv')
    news_words, news_labels = load_external_dataset('data/cwi2018_news_test.tsv')

    print(f"Biomedical: {len(biomed_words)} words, "
          f"{sum(biomed_labels)/len(biomed_labels):.1%} complex")
    print(f"News: {len(news_words)} words, "
          f"{sum(news_labels)/len(news_labels):.1%} complex")


### 5.2 Evaluate Your Model

Apply your best model from Section 4 to both external datasets **without retraining**. Use the same features, the same normalization (training mean/std), and the same classifier.

Report precision, recall, and F1 for each dataset alongside your dev set performance. **Report these metrics in Answer 5.3.**

In [ ]:
if __name__ == "__main__":
    # Feature extraction and normalization; model label prediction
    biomed_feats_all = custom_feat_extraction(biomed_words)
    _, standardized_biomed_feats, _, _ = normalize_features(training_features, biomed_feats_all)
    biomed_predictions = gbc_all.predict(standardized_biomed_feats)
    news_feats_all = custom_feat_extraction(news_words)
    _, standardized_news_feats, _, _ = normalize_features(training_features, news_feats_all)
    news_predictions = gbc_all.predict(standardized_news_feats)

    bio_p, bio_r, bio_f1 = get_precision(biomed_labels, biomed_predictions), get_recall(biomed_labels,
                                    biomed_predictions), get_fscore(biomed_labels, biomed_predictions)
    news_p, news_r, news_f1 = get_precision(news_labels, news_predictions), get_recall(news_labels,
                                    news_predictions), get_fscore(news_labels, news_predictions)
    print(f"{'Dataset':20s} {'P':>8s} {'R':>8s} {'F1':>8s}")
    print(f"{'HW01 Dev':20s} {dev_p:>8.4f} {dev_r:>8.4f} {dev_f1:>8.4f}")
    print(f"{'Biomedical':20s} {bio_p:>8.4f} {bio_r:>8.4f} {bio_f1:>8.4f}")
    print(f"{'News':20s} {news_p:>8.4f} {news_r:>8.4f} {news_f1:>8.4f}")


### 5.3 Domain Transfer Analysis (10 points) [WRITEUP]

In your writeup (Answer 5.3), report **all of the following**:

1. **Class distributions** (from 5.1): Report the % complex for the biomedical and news datasets.

2. **Model performance** (from 5.2): Report precision, recall, and F1 for each dataset alongside your dev set performance.

3. **Transfer analysis:**
   - Why does your model transfer better to one domain than the other? What properties of each domain make classification easier or harder?
   - Provide at least 2 specific examples of words the model gets wrong in the new domains, and explain why those errors occur.

In [ ]:
if __name__ == "__main__":
    # Find errors in predictions for each external dataset
    tps = [w for w, true, pred
           in zip(biomed_words, biomed_labels, biomed_predictions)
           if true == 1 and pred == 1]
    tns = [w for w, true, pred
           in zip(biomed_words, biomed_labels, biomed_predictions)
           if true == 0 and pred == 0]
    false_positives = [w for w, true, pred
                in zip(biomed_words, biomed_labels, biomed_predictions)
                if true == 0 and pred == 1]
    false_negatives = [w for w, true, pred
                in zip(biomed_words, biomed_labels, biomed_predictions)
                if true == 1 and pred == 0]
    print(f"Biomedical: predicted complex, actually complex "
          f"{tps[:10]}")
    print(f"Biomedical: predicted simple, actually simple "
          f"{tns[:10]}")
    print(f"Biomedical: predicted complex, actually simple "
          f"{false_positives[:10]}")
    print(f"Biomedical: predicted simple, actually complex "
          f"{false_negatives[:10]}")
    tps = [w for w, true, pred
           in zip(news_words, news_labels, news_predictions)
           if true == 1 and pred == 1]
    tns = [w for w, true, pred
           in zip(news_words, news_labels, news_predictions)
           if true == 0 and pred == 0]
    false_positives = [w for w, true, pred
                in zip(news_words, news_labels, news_predictions)
                if true == 0 and pred == 1]
    false_negatives = [w for w, true, pred
                in zip(news_words, news_labels, news_predictions)
                if true == 1 and pred == 0]
    print(f"News: predicted complex, actually complex "
          f"{tps[:10]}")
    print(f"News: predicted simple, actually simple "
          f"{tns[:10]}")
    print(f"News: predicted complex, actually simple "
          f"{false_positives[:10]}")
    print(f"News: predicted simple, actually complex "
          f"{false_negatives[:10]}")
    # I don't think "protect", "rebel", or "defend" are complex. Why gold?

---
## Section 6: Reflection Questions (10 points) [WRITEUP]

These questions connect the homework to broader concepts from Module 1.

**[WRITEUP] Answer 6.1 (5 points): Class Imbalance and Metric Choice**

Consider the all-complex baseline from Section 2.1.

1. What was the accuracy of the all-complex baseline? Note that the classes are imbalanced, so this number may not be what you'd expect. What does it reveal about the class distribution, and why is accuracy a misleading metric for this task?
2. Why is F1 score a better metric than accuracy for this task? In your answer, discuss what precision and recall each measure and why both matter for complex word identification.

*Hint: See Chapter 4, §4.9 (Evaluation: Precision, Recall, F-measure) for background on evaluation metrics.*

In [ ]:
if __name__=="__main__":
    # Calculate accuracy for the all_complex baseline on dev data
    all_comp_dev, all_comp_bio, all_comp_news = np.ones(len(dev_labels)), \
            np.ones(len(biomed_labels)), np.ones(len(news_labels))
    print("Dev set precision/recall/F1: ", get_precision(dev_labels, all_comp_dev),
          get_recall(dev_labels, all_comp_dev), get_fscore(dev_labels, all_comp_dev))
    print("Biomedical precision/recall/F1: ", get_precision(biomed_labels, all_comp_bio),
          get_recall(biomed_labels, all_comp_bio), get_fscore(biomed_labels, all_comp_bio))
    print("News precision/recall/F1: ", get_precision(news_labels, all_comp_news),
          get_recall(news_labels, all_comp_news), get_fscore(news_labels, all_comp_news))

**[WRITEUP] Answer 6.2 (5 points): Model Interpretation**

One advantage of logistic regression over more complex models is **interpretability** — we can inspect the learned weights to understand what the model has learned.

1. After training your logistic regression model (Section 3.2), inspect its weights using `clf.coef_` and `clf.intercept_`. Which feature has the **largest positive weight** (indicating "complex")? Which has the **largest negative weight** (indicating "simple")? What do these weights tell you about the relationship between each feature and word complexity?
2. Would you be able to do this kind of interpretation with a Random Forest or neural network? Why is interpretability important in NLP applications? Give one real-world example where a non-interpretable model could cause harm.

*Hint: See Chapter 4, §4.13 (Interpreting Models) and §4.12 (Avoiding Harms in Classification) for relevant background.*

In [ ]:
if __name__ == "__main__":
    # Inspect logistic regression weights for the LR model in section 3
    print(lr_coeffs)  # Features were word length, frequency, and num syllables
    print(lr_intercepts)

---
## Submission Checklist

Before submitting to Gradescope, verify:

- [ ] All code cells run without errors (Kernel → Restart & Run All)
- [ ] `hw01_writeup.pdf` contains answers to all **[WRITEUP]** questions:
  - Answer 2.1 (all-complex baseline metrics + class imbalance discussion)
  - Answer 2.2 (word length baseline + threshold choice)
  - Answer 2.3 (word frequency baseline + baseline comparison)
  - Answer 3.3 (classifier comparison: NB vs LR vs baselines)
  - Answer 4.1 (custom model: feature design + justification)
  - Answer 4.2 (error analysis: examples + error categories)
  - Answer 5.3 (domain transfer analysis: F1 table + explanation + examples)
  - Answer 6.1 (class imbalance and metric choice)
  - Answer 6.2 (model interpretation)
- [ ] `test_labels.txt` contains your test set predictions (one label per line)
- [ ] All files are named exactly: `hw01.ipynb`, `hw01.py`, `hw01_writeup.pdf`, `test_labels.txt`